# Orbit Wars - 試行錯誤の記録

## 1. 環境のセットアップ

In [12]:
%%capture
!pip install --upgrade "kaggle-environments>=1.28.0"

from kaggle_environments import make
import math

env = make("orbit_wars", debug=True)
print(f"Environment: {env.name} v{env.version}")
print(f"Players: {env.specification.agents}")
print(f"Max steps: {env.configuration.episodeSteps}")

## 2. ゲーム理解

In [13]:
# Run a quick game to see what the observation looks like
env = make("orbit_wars", debug=True)
env.run(["random", "random"])

# Peek at the initial observation
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet

obs = env.steps[1][0].observation  # step 1 = first action step
planets = [Planet(*p) for p in obs.planets]
print(f"Player: {obs.player}")
print(f"Angular velocity: {obs.angular_velocity:.4f} rad/turn")
print(f"\nPlanets ({len(planets)}):")
for p in planets[:6]:
    owner_str = f"Player {p.owner}" if p.owner >= 0 else "Neutral"
    print(f"  id={p.id} owner={owner_str:10s} pos=({p.x:.1f}, {p.y:.1f}) r={p.radius:.1f} ships={p.ships} prod={p.production}")

Player: 0
Angular velocity: 0.0464 rad/turn

Planets (32):
  id=0 owner=Neutral    pos=(95.0, 75.9) r=2.1 ships=8 prod=3
  id=1 owner=Neutral    pos=(24.1, 95.0) r=2.1 ships=8 prod=3
  id=2 owner=Neutral    pos=(75.9, 5.0) r=2.1 ships=8 prod=3
  id=3 owner=Neutral    pos=(5.0, 24.1) r=2.1 ships=8 prod=3
  id=4 owner=Neutral    pos=(83.1, 96.7) r=1.7 ships=17 prod=2
  id=5 owner=Neutral    pos=(3.3, 83.1) r=1.7 ships=17 prod=2


## 3. エージェントの実装

In [14]:
# ============================================================
# agent_v1：一番近い惑星を狙うだけの基本エージェント
# スコア：約400
# 問題点：距離だけで判断するため生産力の低い惑星を狙いがち
# ============================================================
def agent_v1(obs, config=None):
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]
    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]
    if not targets:
        return moves
    for mine in my_planets:
        nearest = min(targets, key=lambda t: math.hypot(mine.x-t.x, mine.y-t.y))
        ships_needed = max(nearest.ships + 1, 20)
        if mine.ships >= ships_needed:
            angle = math.atan2(nearest.y-mine.y, nearest.x-mine.x)
            moves.append([mine.id, angle, ships_needed])
    return moves

In [15]:
# ============================================================
# agent_v2：距離÷生産力で優先度をつけたエージェント
# スコア：465
# 改善点：生産力が高くて近い惑星を優先するようにした
# 改善点：守備用に最低10隻（GUARD）を残すようにした
# 問題点：最低20隻送るため攻撃が遅い
# ============================================================
def agent_v2(obs, config=None):
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]
    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]
    if not targets:
        return moves
    for mine in my_planets:
        def priority(t):
            dist = math.hypot(mine.x-t.x, mine.y-t.y)
            return dist / (t.production + 1)
        nearest = min(targets, key=priority)
        GUARD = 10
        ships_needed = max(nearest.ships + 1, 20)
        if mine.ships >= ships_needed + GUARD:
            angle = math.atan2(nearest.y-mine.y, nearest.x-mine.x)
            moves.append([mine.id, angle, ships_needed])
    return moves

In [16]:
# ============================================================
# agent_v3：静止惑星のみを狙うエージェント
# スコア：600前後
# 改善点：自転惑星を狙うと艦隊がすり抜けることを発見
#         → 静止惑星のみに絞ることで艦隊の無駄をなくした
# 問題点：自転惑星（内側の生産力高い惑星）を全く取れない
#         → 実際の対戦では内側を先に取られて負けることが多い
# ============================================================
def agent_v3(obs, config=None):
    import math
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]
    angular_velocity = obs.get("angular_velocity", 0)

    my_planets = [p for p in planets if p.owner == player]
    
    cx, cy = 50, 50
    # 静止惑星だけを狙う
    targets = [p for p in planets 
               if p.owner != player
               and math.hypot(p.x-cx, p.y-cy) + p.radius >= 50]

    if not targets:
        # 静止惑星がなければ全部対象
        targets = [p for p in planets if p.owner != player]

    for mine in my_planets:
        def priority(t):
            dist = math.hypot(mine.x-t.x, mine.y-t.y)
            return dist / (t.production + 1)
        nearest = min(targets, key=priority)
        GUARD = 10
        ships_needed = max(nearest.ships + 1, 20)
        if mine.ships >= ships_needed + GUARD:
            angle = math.atan2(nearest.y-mine.y, nearest.x-mine.x)
            moves.append([mine.id, angle, ships_needed])

    return moves

In [17]:
# ============================================================
# agent_v4：自転惑星の未来位置予測 + 少数艦隊で素早く広げる
# スコア：650前後（現在の最強版）
# 改善点①：ships=8に下げて素早く多くの惑星を確保できるようにした
# 改善点②：自転惑星の到着時の位置を予測して狙うようにした
# ============================================================
def agent_v4(obs, config=None):
    import math
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]
    angular_velocity = obs.get("angular_velocity", 0)

    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]

    if not targets:
        return moves

    cx, cy = 50, 50

    for mine in my_planets:
        def priority(t):
            dist = math.hypot(mine.x-t.x, mine.y-t.y)
            return dist / (t.production + 1)

        nearest = min(targets, key=priority)
        ships_needed = max(nearest.ships + 1, 8)

        # 自転惑星かどうか判定
        dist_from_sun = math.hypot(nearest.x-cx, nearest.y-cy)
        is_orbiting = dist_from_sun + nearest.radius < 50

        if is_orbiting:
            # 速度3で到着ターン数を推定
            turns = int(math.hypot(mine.x-nearest.x, mine.y-nearest.y) / 3) + 1
            dx = nearest.x - cx
            dy = nearest.y - cy
            r = math.sqrt(dx**2 + dy**2)
            cur_angle = math.atan2(dy, dx)
            fut_angle = cur_angle + angular_velocity * turns
            target_x = cx + r * math.cos(fut_angle)
            target_y = cy + r * math.sin(fut_angle)
        else:
            target_x, target_y = nearest.x, nearest.y

        GUARD = 10
        if mine.ships >= ships_needed + GUARD:
            angle = math.atan2(target_y - mine.y, target_x - mine.x)
            moves.append([mine.id, angle, ships_needed])

    return moves

## 4. 実装後の評価

### バージョンアップ前後のエージェントの対戦結果

In [19]:
v2_wins = 0
for i in range(10):
    env = make("orbit_wars", configuration={"seed": i}, debug=False)
    env.run([agent_v2, agent_v1])
    result = env.steps[-1]
    if result[0].reward > result[1].reward:
        v2_wins += 1
print(f"v2 vs v1: {v2_wins}/10")

v2 vs v1: 7/10


In [20]:
v3_wins = 0
for i in range(10):
    env = make("orbit_wars", configuration={"seed": i}, debug=False)
    env.run([agent_v3, agent_v2])
    result = env.steps[-1]
    if result[0].reward > result[1].reward:
        v3_wins += 1
print(f"v3 vs v2: {v3_wins}/10")

v3 vs v2: 10/10


In [21]:
v4_wins = 0
for i in range(10):
    env = make("orbit_wars", configuration={"seed": i}, debug=False)
    env.run([agent_v4, agent_v3])
    result = env.steps[-1]
    if result[0].reward > result[1].reward:
        v4_wins += 1
print(f"v4 vs v3: {v4_wins}/10")

v4 vs v3: 8/10


### 分析
勝率比較からエージェントの性能がバージョンアップによって改善されていることがわかる。

しかし、スコアが700前後で安定してしまう。

v4時点での問題点
* 自転する惑星をほとんど取れずに負ける→内側は生産力が高いため
* 太陽にあたって自滅する場合がある→太陽回避の動きを組み込むと逆に性能が悪化した

v4の対戦を見て得た気づき
* 自転惑星もたまたまあたって取れることがある→この場合は勝ちやすい
* 外側惑星の生産力が低いマップは負けやすい→生産力の差で負ける

## 5. submission.pyの出力

In [22]:
%%writefile submission.py
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet

def agent(obs, config=None):
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]

    my_planets = [p for p in planets if p.owner == player]
    
    cx, cy = 50, 50
    # 静止惑星だけを狙う
    targets = [p for p in planets 
               if p.owner != player
               and math.hypot(p.x-cx, p.y-cy) + p.radius >= 50]

    if not targets:
        # 静止惑星がなければ全部対象
        targets = [p for p in planets if p.owner != player]

    for mine in my_planets:
        def priority(t):
            dist = math.hypot(mine.x-t.x, mine.y-t.y)
            return dist / (t.production + 1)
        nearest = min(targets, key=priority)
        GUARD = 10
        ships_needed = max(nearest.ships + 1, 8)
        if mine.ships >= ships_needed + GUARD:
            angle = math.atan2(nearest.y-mine.y, nearest.x-mine.x)
            moves.append([mine.id, angle, ships_needed])

    return moves

Writing submission.py
